# KoBART Summarizer 재학습 (Google Colab)

lotte-insight 프로젝트 — `training/train_summarizer.py` Colab 실행용 노트북

**학습 타깃:** `event_summary` 평문 한국어 문장 (JSON 아님)  
**평가 지표:** `eval_loss` (best model 선택 기준), `char_f1`, `exact_match`

In [ ]:
# 1. GPU 확인
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Not available — 런타임을 GPU로 변경하세요')
print('CUDA:', torch.version.cuda)

In [ ]:
# 2. 레포 클론 및 의존성 설치
# GitHub URL을 본인 레포로 교체하시오
GITHUB_REPO_URL = 'https://github.com/JoeYunHa/Lotte_Insight'

!git clone {GITHUB_REPO_URL} /content/lotte-insight
%cd /content/lotte-insight/training
!pip install -q -r requirements.txt

In [ ]:
# 3. 학습 데이터 업로드 (로컬 → Colab 직접 전송)
# 실행 후 파일 선택 창에서 아래 파일을 선택하시오:
#   - labeled_titles.csv
#   - labeled_players.csv
#   - game_results.csv  (경기 결과 컨텍스트용 — game_context 컬럼이 CSV에 없으면 필수)
import os
import pandas as pd
from google.colab import files

DATA_DIR = '/content/lotte-insight/training/data'
os.makedirs(DATA_DIR, exist_ok=True)

uploaded = files.upload()  # 파일 선택 창 열림

for fname, content in uploaded.items():
    dst = f'{DATA_DIR}/{os.path.basename(fname)}'
    with open(dst, 'wb') as f:
        f.write(content)
    print(f'저장 완료: {dst}  ({len(content):,} bytes)')

if not os.path.exists(f'{DATA_DIR}/game_results.csv'):
    print('[WARN] game_results.csv 없음 — game_context 컬럼이 CSV에 채워져 있지 않으면 MATCH_RELATED 요약 품질 저하')

print()
for fname in ['labeled_titles.csv', 'labeled_players.csv']:
    path = f'{DATA_DIR}/{fname}'
    if not os.path.exists(path):
        print(f'[MISSING] {fname}')
        continue
    df = pd.read_csv(path, encoding='utf-8-sig')
    lotte = df[df['is_lotte_related'].astype(str).str.lower() == 'true']
    with_summary = lotte['event_summary'].fillna('').astype(str).str.strip().ne('').sum()
    with_game_ctx = lotte['game_context'].fillna('').astype(str).str.strip().ne('').sum() if 'game_context' in lotte.columns else 0
    match_rows = lotte[lotte['primary_label'] == 'MATCH_RELATED']
    match_with_ctx = match_rows['game_context'].fillna('').astype(str).str.strip().ne('').sum() if 'game_context' in match_rows.columns else 0
    print(f'{fname}: 전체={len(df)}행  lotte={len(lotte)}행')
    print(f'  event_summary 있음: {with_summary}행 ({with_summary/len(lotte)*100:.0f}% of lotte)')
    print(f'  game_context 있음: {with_game_ctx}행')
    print(f'  MATCH_RELATED 중 game_context: {match_with_ctx}/{len(match_rows)}행')
    if with_summary < len(lotte) * 0.3:
        print(f'  [주의] event_summary 채움률 낮음 — 로컬에서 add_summaries.py 실행 후 재업로드 필요')
    if len(match_rows) > 0 and match_with_ctx < len(match_rows) * 0.5:
        print(f'  [주의] MATCH_RELATED 행 중 game_context 미달 — game_results.csv 업로드 후 add_summaries.py 실행 필요')

print('
샘플 확인 (event_summary 평문):')
for fname in ['labeled_titles.csv', 'labeled_players.csv']:
    path = f'{DATA_DIR}/{fname}'
    if not os.path.exists(path):
        continue
    df = pd.read_csv(path, encoding='utf-8-sig')
    samples = df[df['event_summary'].fillna('').astype(str).str.strip().ne('')]['event_summary'].head(2)
    for s in samples:
        print(f'  {repr(s[:80])}')


In [ ]:
# 3-2. (선택) game_context + event_summary 백필
# game_results.csv를 업로드했고 labeled CSV에 game_context/event_summary가 없거나
# 갱신이 필요할 때만 실행하시오. OpenAI API 키와 네이버 API 키가 필요합니다.
#
# 이미 로컬에서 add_summaries.py를 실행해 CSV에 채워진 경우 이 셀은 건너뛰어도 됩니다.

import os

# OpenAI API 키 설정 (필수)
os.environ['OPENAI_API_KEY'] = 'sk-...'  # 실제 키로 교체

DATA_DIR = '/content/lotte-insight/training/data'

if not os.path.exists(f'{DATA_DIR}/game_results.csv'):
    print('[SKIP] game_results.csv 없음 — game_context 없이 진행됩니다')
else:
    for dataset in ['titles', 'players']:
        csv_name = 'labeled_titles.csv' if dataset == 'titles' else 'labeled_players.csv'
        if not os.path.exists(f'{DATA_DIR}/{csv_name}'):
            continue
        print(f'
[{dataset}] game_context + event_summary 백필 중...')
        !python add_summaries.py --dataset {dataset}
    print('
백필 완료 — game_context/event_summary 채워진 CSV로 학습합니다')


In [ ]:
# 4. 학습 실행
# 타깃: event_summary 평문 (JSON 아님)
# best model 기준: eval_loss 최소
!python train_summarizer.py \
    --epochs 5 \
    --batch 8 \
    --max-source-len 256 \
    --max-target-len 192 \
    --num-beams 4 \
    --early-stopping-patience 2

In [ ]:
# 5. 학습 결과 확인
import json

state_path = '/content/lotte-insight/training/models/summarizer_kobart/trainer_state.json'
with open(state_path) as f:
    state = json.load(f)

print(f'best_metric (eval_loss): {state["best_metric"]:.4f}')
print(f'best_step: {state["best_global_step"]}')
print(f'총 학습 step: {state["global_step"]}')

print('
에폭별 평가 지표:')
for entry in state['log_history']:
    if 'eval_loss' not in entry:
        continue
    rouge_l  = entry.get('eval_rouge_l',  entry.get('rouge_l',  '-'))
    char_f1  = entry.get('eval_char_f1',  entry.get('char_f1',  '-'))
    fmt_rouge = f'{rouge_l:.4f}' if isinstance(rouge_l, float) else rouge_l
    fmt_char  = f'{char_f1:.4f}' if isinstance(char_f1, float) else char_f1
    print(f'  epoch={entry["epoch"]:.0f}  eval_loss={entry["eval_loss"]:.4f}'
          f'  rouge_l={fmt_rouge}  char_f1={fmt_char}')


In [ ]:
# 6. 빠른 추론 테스트 (학습 직후 품질 확인)
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

MODEL_DIR = '/content/lotte-insight/training/models/summarizer_kobart'
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR)
model.eval()

# game_context 포함 샘플: 훈련 시 game_context를 source에 주입했으므로 추론도 동일하게 구성
SAMPLES = [
    # MATCH_RELATED — game_context 포함 (경기 결과 반영 여부 확인)
    'title: 롯데 나균안, 시즌 5승 달성…선발 로테이션 안정화
'
    'description: 나균안이 두산전 6이닝 2실점 호투로 시즌 5승을 따냈다.
'
    'topic_label: MATCH_RELATED
'
    'game_context: 롯데 vs 두산 (홈) 5-2 승 | 결승타: 나균안(6회) | 투수 키플레이어: 나균안(롯데)',
    # MATCH_RELATED — game_context 없음 (비교용)
    'title: 롯데 나균안, 시즌 5승 달성…선발 로테이션 안정화
'
    'description: 나균안이 두산전 6이닝 2실점 호투로 시즌 5승을 따냈다.
'
    'topic_label: MATCH_RELATED',
    # INJURY_ROSTER
    'title: 롯데 전준우 햄스트링 부상…2주 결장 예상
'
    'description: 전준우가 경기 중 햄스트링 부상으로 1군 엔트리에서 말소됐다.
'
    'topic_label: INJURY_ROSTER',
]

for source in SAMPLES:
    inputs = tokenizer(source, max_length=256, truncation=True, return_tensors='pt')
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=192, num_beams=4, no_repeat_ngram_size=3)
    result = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    first_line = source.splitlines()[0]
    has_ctx = 'game_context:' in source
    print(f'[game_context={"있음" if has_ctx else "없음"}] {first_line}')
    print(f'요약: {result}')
    print()


In [ ]:
# 7. 학습된 모델 다운로드 (Colab → 로컬)
import shutil, os
from google.colab import files

LOCAL_MODEL_DIR = '/content/lotte-insight/training/models/summarizer_kobart'
ZIP_PATH = '/content/summarizer_kobart.zip'

if os.path.exists(LOCAL_MODEL_DIR):
    shutil.make_archive('/content/summarizer_kobart', 'zip', LOCAL_MODEL_DIR)
    print(f'압축 완료: {ZIP_PATH}')
    files.download(ZIP_PATH)  # 로컬로 다운로드
else:
    print('[ERROR] 모델 디렉토리 없음 — 학습 실패 여부 확인 필요')

In [ ]:
# 8. (선택) 평가만 실행할 경우
# !python train_summarizer.py --eval-only